# Classroom Voice Analysis

Upload a lesson recording. This transcribes it, works out **who spoke when**, decides **which voice is the teacher**, and reports how the talking was distributed.

```
audio -> normalize -> transcribe -> diarize -> assign -> roles -> analyze
```

Source: https://github.com/JakeOJeff/cva

---

**Set the runtime to a GPU first:** Runtime -> Change runtime type -> T4 GPU.

It runs on CPU too, but a GPU turns hours into minutes, and Colab gives you one free.

## 1. Install

Takes about two minutes.

In [ ]:
!git clone -q https://github.com/JakeOJeff/cva.git
%cd cva
!pip install -q -r requirements.txt 2>&1 | tail -2

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none - this will be slow, switch the runtime to T4")

## 2. HuggingFace token

Speaker diarization uses a gated model, so it needs a free read token.

1. Accept the terms once at [pyannote/speaker-diarization-community-1](https://hf.co/pyannote/speaker-diarization-community-1) — the page must say *"You have been granted access"*
2. Create a **read** token at [hf.co/settings/tokens](https://hf.co/settings/tokens)

A fine-grained token needs *"Read access to contents of all public gated repos"* ticked. A plain Read token works without it.

In [ ]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("HF token (hidden): ").strip()

# Fails now with a clear reason rather than 40 minutes into a transcription.
from pipeline import diarize
diarize.get_pipeline()
print("access OK - diarization will run")

## 3. Pick audio

Run this and upload your own file, or skip it to use the bundled sample lesson.

In [ ]:
from google.colab import files

AUDIO = "assets/audio.mp3"      # the bundled 64-minute Hindi lesson
LANGUAGE = "hi"                 # hi, ml, ta, te, kn, en
MINUTES = 5                     # how much of it to process; None for all of it

up = files.upload()             # cancel this to keep the bundled sample
if up:
    AUDIO = list(up)[0]
print("using:", AUDIO)

In [ ]:
from pipeline import audio

samples = audio.decode(AUDIO)
print(f"{len(samples)/16000/60:.1f} minutes of audio")

if MINUTES:
    samples = samples[: 16000 * 60 * MINUTES]
    print(f"trimmed to {len(samples)/16000/60:.1f} minutes")

CLIP = "clip.wav"
audio.write_wav(samples, CLIP)

## 4. Check the language before spending time on it

Whisper does **not** error on the wrong language. It returns fluent nonsense, or nothing at all, and every number downstream is then computed faithfully over that nonsense. So check first.

In [ ]:
from pipeline import transcribe

detected, confidence, windows = transcribe.detect_language(CLIP)
print(f"sounds like '{detected}' (confidence {confidence})")
print(f"you chose   '{LANGUAGE}'")
if detected != LANGUAGE and confidence >= 0.5:
    print("\n  MISMATCH - set LANGUAGE above and re-run, or the transcript will be nonsense.")
else:
    print("\n  agrees.")

## 5. Run it

`small` is the smallest model that can actually write Indian-language scripts — `tiny` emits confident English instead. On a T4 this is minutes.

In [ ]:
import sys, time
from pipeline import run

start = time.time()

def progress(stage, fraction, note):
    bar = "#" * int(fraction * 24)
    sys.stdout.write(f"\r  {stage:<11} [{bar:<24}] {fraction:4.0%}  {note[:40]:<40}")
    if fraction >= 1.0:
        sys.stdout.write("\n")

result = run.run(
    CLIP,
    work_dir="out",
    language=LANGUAGE,
    model_size="small",
    max_speakers=6,
    use_llm=False,          # the AI review costs money; metrics are free
    on_progress=progress,
)
print(f"\ndone in {time.time()-start:.0f}s")

## 6. Who was in the room

In [ ]:
v = result["teacher"]
print(f"teacher: {v['teacher']}  ({v['confidence']} confidence, margin {v['margin']})")
for reason in v["reasons"]:
    print("   -", reason)

print()
for sp, s in result["speakers"].items():
    print(f"  {s['label']:<13} {sp:<12} {s['talk_share']:>6.1%}  "
          f"{s['n_turns']:>4} turns  avg {s['avg_turn']:>5.1f}s")

## 7. What the numbers say

In [ ]:
m = result["metrics"]

if result["meta"].get("wrong_script"):
    print("!! This transcript is not in the right script - the model was too")
    print("   small for this language. Talk ratios and the timeline still hold")
    print("   (they come from the audio); the words and question counts do not.\n")

rows = [
    ("teacher talk",      f"{m['teacher_talk_ratio']:.0%} of speech"),
    ("student talk",      f"{m['student_talk_ratio']:.0%} of speech"),
    ("silence",           f"{m['silence_ratio']:.0%} of the lesson"),
    ("teacher questions", f"{m['teacher_questions']} ({m['questions_per_10min']}/10min)"),
    ("answered",          f"{m['questions_answered']} ({m['question_response_rate']:.0%})"),
    ("median wait time",  "n/a" if m["median_wait_time"] is None else f"{m['median_wait_time']:.1f}s"),
    ("longest monologue", f"{m['teacher_longest_turn']:.0f}s"),
    ("IRF triads",        m["irf_triads"]),
    ("students heard",    m["n_students_heard"]),
]
for k, val in rows:
    print(f"  {k:<20} {val}")

## 8. Who spoke when

Teacher on top, students below, time along the bottom.

In [ ]:
import matplotlib.pyplot as plt

utts = result["utterances"]
order = [result["teacher"]["teacher"]] + [
    s for s in result["speakers"] if s != result["teacher"]["teacher"]]
row = {sp: i for i, sp in enumerate(order)}
# Validated colourblind-safe categorical order; teacher always takes slot 1.
palette = ["#b4531f", "#1668c4", "#3f7d2a", "#8b46b0", "#8a6d00"]

fig, ax = plt.subplots(figsize=(13, 0.6 * len(order) + 1.4))
for u in utts:
    i = row.get(u["speaker"], len(order) - 1)
    ax.barh(i, u["end"] - u["start"], left=u["start"], height=0.62,
            color=palette[i % len(palette)], edgecolor="none")

ax.set_yticks(range(len(order)))
ax.set_yticklabels([result["speakers"][s]["label"] for s in order])
ax.invert_yaxis()
ax.set_xlabel("seconds into the lesson")
ax.set_xlim(0, result["meta"]["duration"])
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 9. The transcript

In [ ]:
def mmss(t):
    return f"{int(t//60):02d}:{int(t%60):02d}"

for u in result["utterances"]:
    mark = " " if u["speaker_conf"] >= 0.75 else "~"   # ~ = straddled a speaker change
    print(f"[{mmss(u['start'])}]{mark}{u['label']:<12} {u['text']}")

## 10. Correct the teacher, if it got it wrong

Every stage is a pure function over the one before it, so this re-derives every number **without touching the audio again** — about a second, rather than re-running the whole pipeline.

Set the speaker id and run.

In [ ]:
CORRECT_TEACHER = None      # e.g. "SPEAKER_01"

if CORRECT_TEACHER:
    import time
    t = time.time()
    result = run.reanalyze("out", language=LANGUAGE, use_llm=False,
                           teacher=CORRECT_TEACHER)
    print(f"re-derived in {time.time()-t:.2f}s")
    print("teacher talk ratio is now", f"{result['metrics']['teacher_talk_ratio']:.0%}")
else:
    print("set CORRECT_TEACHER above to override")

## 11. Take the results with you

In [ ]:
from pipeline import analyze

with open("out/transcript.txt", "w", encoding="utf-8") as f:
    f.write(analyze.build_transcript(result["utterances"]))

files.download("out/result.json")
files.download("out/transcript.txt")